# Face Mask Detector — Train on Google Colab (GPU)

Trains the MobileNetV2 mask classifier on a free GPU and gives you
**`mask_detector.h5`** + **`plot.png`** to download back to your PC.

**How to run:** `Runtime → Change runtime type → GPU`, then `Runtime → Run all`.
Nothing to upload — the dataset and the Keras-2 compatibility layer install
themselves.

**Why Keras 2?** Colab now ships TensorFlow 2.17+ with Keras 3, which **removed**
`ImageDataGenerator`. The first code cell installs `tf-keras` and sets
`TF_USE_LEGACY_KERAS=1` so the proven Keras-2 training code runs unchanged, and
the resulting `.h5` loads cleanly with the local detection scripts.

## 1. Check the GPU
If this errors or shows no GPU, set `Runtime → Change runtime type → GPU` first.

In [1]:
!nvidia-smi

## 2. Setup + get the dataset (auto, no login)

Installs deps and pulls ~3,800 balanced images into `dataset/`. It tries two
public mirrors, so one repo being down doesn't block you, and it verifies the
`with_mask` / `without_mask` folders actually arrived before moving on.

In [2]:
# Install a Keras-2 layer matching the installed TensorFlow (+ imutils).
# Colab's Keras 3 removed ImageDataGenerator, which the training code needs.
import importlib.metadata as _md, os, shutil, subprocess, sys
try:
    _mm = '.'.join(_md.version('tensorflow').split('.')[:2])
    _spec = 'tf-keras~=' + _mm + '.0'
except Exception:
    _spec = 'tf-keras'
print('[INFO] installing', _spec)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', _spec, 'imutils'])

# Same with_mask/without_mask layout in both mirrors.
MIRRORS = [
    'https://github.com/chandrikadeb7/Face-Mask-Detection.git',
    'https://github.com/balajisrinivas/Face-Mask-Detection.git',
]

def have_dataset():
    return all(
        os.path.isdir(os.path.join('dataset', c)) and
        len(os.listdir(os.path.join('dataset', c))) > 0
        for c in ('with_mask', 'without_mask'))

if not have_dataset():
    for url in MIRRORS:
        print('[INFO] fetching dataset from', url)
        shutil.rmtree('repo', ignore_errors=True)
        if subprocess.call(['git', 'clone', '--depth', '1', url, 'repo']) != 0:
            print('[WARN] clone failed, trying next mirror...')
            continue
        if os.path.isdir('repo/dataset'):
            shutil.rmtree('dataset', ignore_errors=True)
            shutil.copytree('repo/dataset', 'dataset')
        if have_dataset():
            print('[INFO] dataset ready from', url)
            break

assert have_dataset(), (
    'Could not fetch the dataset from any mirror. Check GitHub access, or '
    'upload a dataset/ folder with with_mask/ and without_mask/ subfolders.')

for c in sorted(os.listdir('dataset')):
    p = os.path.join('dataset', c)
    if os.path.isdir(p):
        print(c, '->', len(os.listdir(p)), 'images')

## 3. Train the model

Same architecture and hyper-parameters as `train_mask_detector.py`. After
saving, it reloads the `.h5` to prove the file is valid before you leave Colab.

In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'  # use Keras 2 API (must precede TF import)

import matplotlib
matplotlib.use('Agg')

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from tensorflow.keras.layers import AveragePooling2D, Dropout, Flatten, Dense, Input
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from imutils import paths
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

# Confirm we are on a GPU. If this prints [], training will be 30-60x slower on
# the CPU -> stop and set Runtime -> Change runtime type -> T4 GPU.
gpus = tf.config.list_physical_devices('GPU')
print('[INFO] GPUs visible to TensorFlow:', gpus)
if not gpus:
    print('[WARN] No GPU detected! Training will be very slow on CPU. '
          'Set Runtime -> Change runtime type -> T4 GPU and re-run.')

INIT_LR = 1e-4
EPOCHS = 20
BS = 32
DATASET = 'dataset'

print('[INFO] loading images...')
imagePaths = list(paths.list_images(DATASET))
assert len(imagePaths) > 0, 'No images found under dataset/ — re-run cell 2.'

data, labels = [], []
for imagePath in imagePaths:
    label = imagePath.split(os.path.sep)[-2]
    image = load_img(imagePath, target_size=(224, 224))
    image = img_to_array(image)
    image = preprocess_input(image)
    data.append(image)
    labels.append(label)

data = np.array(data, dtype='float32')
labels = np.array(labels)
assert len(np.unique(labels)) == 2, 'Expected 2 classes, found: ' + str(np.unique(labels))
print('[INFO] loaded', len(data), 'images across classes', list(np.unique(labels)))

lb = LabelBinarizer()
labels = lb.fit_transform(labels)
labels = to_categorical(labels)

(trainX, testX, trainY, testY) = train_test_split(
    data, labels, test_size=0.20, stratify=labels, random_state=42)

aug = ImageDataGenerator(rotation_range=20, zoom_range=0.15,
    width_shift_range=0.2, height_shift_range=0.2, shear_range=0.15,
    horizontal_flip=True, fill_mode='nearest')

baseModel = MobileNetV2(weights='imagenet', include_top=False,
    input_tensor=Input(shape=(224, 224, 3)))
headModel = baseModel.output
headModel = AveragePooling2D(pool_size=(7, 7))(headModel)
headModel = Flatten(name='flatten')(headModel)
headModel = Dense(128, activation='relu')(headModel)
headModel = Dropout(0.5)(headModel)
headModel = Dense(2, activation='softmax')(headModel)
model = Model(inputs=baseModel.input, outputs=headModel)
for layer in baseModel.layers:
    layer.trainable = False

print('[INFO] compiling model...')
opt = Adam(learning_rate=INIT_LR)
model.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])

print('[INFO] training head...')
# verbose=2 prints ONE line per epoch (the live progress bar does not render in
# the VS Code output panel, which makes it look frozen).
H = model.fit(
    aug.flow(trainX, trainY, batch_size=BS),
    steps_per_epoch=len(trainX) // BS,
    validation_data=(testX, testY),
    validation_steps=len(testX) // BS,
    epochs=EPOCHS,
    verbose=2)

print('[INFO] evaluating network...')
predIdxs = np.argmax(model.predict(testX, batch_size=BS), axis=1)
print(classification_report(testY.argmax(axis=1), predIdxs,
    target_names=lb.classes_))

model.save('mask_detector.h5')
print('[INFO] saved mask_detector.h5')

# Sanity check: reload the file we just wrote so we know it is valid.
reloaded = load_model('mask_detector.h5')
print('[INFO] reload OK — input', reloaded.input_shape, 'output', reloaded.output_shape)

N = EPOCHS
plt.style.use('ggplot')
plt.figure()
plt.plot(np.arange(0, N), H.history['loss'], label='train_loss')
plt.plot(np.arange(0, N), H.history['val_loss'], label='val_loss')
plt.plot(np.arange(0, N), H.history['accuracy'], label='train_acc')
plt.plot(np.arange(0, N), H.history['val_accuracy'], label='val_acc')
plt.title('Training Loss and Accuracy')
plt.xlabel('Epoch #')
plt.ylabel('Loss/Accuracy')
plt.legend(loc='lower left')
plt.savefig('plot.png')
print('[INFO] saved plot.png')

## 4. Download the results to your PC

Drop `mask_detector.h5` into your project folder, then run
`python detect_mask_video.py` locally.

In [ ]:
from google.colab import files
files.download('mask_detector.h5')
files.download('plot.png')